# Fine-tune and Evaluate OPT-1.3B

### Training and validation

Only complete instruction, article and structured-target pairs are kept.
Targets may use up to 450 tokens, and the complete pair must fit within 2,048
tokens. Nothing is truncated.

### Testing and deployment

```text
Instruction + article <= 1,598 tokens
    → direct structured generation

Otherwise
    → sentence-based article chunks of at most 450 tokens
    → short factual notes for each chunk
    → recursively shorten the combined notes when needed
    → final structured generation
```

The raw model response is normalized into one fixed structure for every test
record. The `generated_summary` column stores only the Summary section; the
full normalized response remains in `generated_text`.


## 0. Setup

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import re
import json
import time
import random
from functools import partial

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from rouge_score import rouge_scorer
import textstat

PROCESSED_DIR = "data/processed"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-1.3b"
FINETUNED_MODEL_DIR = "models/opt-1.3b-cnn-news-simplifier"
RESULTS_DIR = "results"
SEED = 42

MAX_CONTEXT = 2048
MAX_TRAIN_LENGTH = 2048
MAX_TARGET_TOKENS = 450

MAX_NEW_TOKENS = 450
DIRECT_INPUT_TOKENS = 1598
CHUNK_ARTICLE_TOKENS = 450
CHUNK_NEW_TOKENS = 96
NOTES_REDUCTION_NEW_TOKENS = 64

if DIRECT_INPUT_TOKENS + MAX_NEW_TOKENS != MAX_CONTEXT:
    raise ValueError("Direct input and output budgets must total 2,048 tokens.")

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before running this notebook.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda"
print(f"Using device: {DEVICE}")


## 1. Load train, validation and test splits

In [ ]:
def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Run the optimized preprocessing and "
            "split notebooks first."
        )

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return pd.DataFrame(records)


train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print(f"Loaded training:   {len(train_df)}")
print(f"Loaded validation: {len(val_df)}")
print(f"Loaded test:       {len(test_df)}")


## 2. Keep only complete training pairs

The token lengths are recomputed here. Training and validation keep only
complete targets of at most 450 tokens and complete prompt-target pairs of at
most 2,048 tokens. The dataset class checks the same rules again and never
uses truncation.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def encoded_length(text: str, add_special_tokens: bool) -> int:
    return len(
        tokenizer(
            text,
            add_special_tokens=add_special_tokens,
            truncation=False,
        )["input_ids"]
    )


def keep_complete_pairs(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    frame = frame.copy()
    frame["prompt_token_length"] = frame["prompt"].apply(
        lambda text: encoded_length(str(text), add_special_tokens=True)
    )
    frame["target_token_length"] = frame["training_target"].apply(
        lambda text: encoded_length(
            str(text).strip() + tokenizer.eos_token,
            add_special_tokens=False,
        )
    )
    frame["total_training_tokens"] = (
        frame["prompt_token_length"] + frame["target_token_length"]
    )

    complete_mask = (
        (frame["target_token_length"] <= MAX_TARGET_TOKENS) &
        (frame["total_training_tokens"] <= MAX_TRAIN_LENGTH)
    )
    complete = frame.loc[complete_mask].reset_index(drop=True)

    print(
        f"{split_name}: kept {len(complete)}/{len(frame)} complete pairs; "
        f"removed {len(frame) - len(complete)} non-fitting pairs"
    )

    if complete.empty:
        raise ValueError(f"No complete {split_name.lower()} pairs remain.")

    return complete


train_df = keep_complete_pairs(train_df, "Training")
val_df = keep_complete_pairs(val_df, "Validation")


class NewsSimplificationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        max_length: int = MAX_TRAIN_LENGTH,
        max_target_tokens: int = MAX_TARGET_TOKENS,
    ):
        self.prompts = frame["prompt"].tolist()
        self.targets = frame["training_target"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_target_tokens = max_target_tokens

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        prompt = str(self.prompts[index])
        target = str(self.targets[index]).strip() + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
        target_ids = self.tokenizer(
            target,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        if len(target_ids) > self.max_target_tokens:
            raise ValueError(
                f"Target {index} has {len(target_ids)} tokens; "
                f"maximum is {self.max_target_tokens}."
            )

        total_length = len(prompt_ids) + len(target_ids)
        if total_length > self.max_length:
            raise ValueError(
                f"Training pair {index} has {total_length} tokens; "
                f"maximum is {self.max_length}."
            )

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


def collate_fn(batch, pad_token_id: int):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        pad_length = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_token_id] * pad_length)
        attention_masks.append(item["attention_mask"] + [0] * pad_length)
        labels.append(item["labels"] + [-100] * pad_length)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


train_dataset = NewsSimplificationDataset(train_df, tokenizer)
val_dataset = NewsSimplificationDataset(val_df, tokenizer)

print(
    f"Prepared {len(train_dataset)} training examples and "
    f"{len(val_dataset)} validation examples"
)


## 3. Load OPT-1.3B and fine-tune with QLoRA

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)

print("Model context limit:", model.config.max_position_embeddings)

if MAX_TRAIN_LENGTH > model.config.max_position_embeddings:
    raise ValueError(
        "MAX_TRAIN_LENGTH exceeds the model context limit."
    )

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(FINETUNED_MODEL_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

safe_collator = partial(
    collate_fn,
    pad_token_id=tokenizer.pad_token_id,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=safe_collator,
)

train_result = trainer.train()
print(train_result)


## 4. Save the fine-tuned adapter

In [ ]:
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)
print(f"Saved model adapter to '{FINETUNED_MODEL_DIR}'")

## 5. Deterministic direct or chunk-based test generation

- Direct generation is used only when the complete instruction and article
  prompt is at most 1,598 tokens.
- Longer articles are split into sentence-based chunks of at most 450 article
  tokens. A rare sentence longer than 450 tokens is divided into token pieces
  without dropping any tokens.
- Each chunk produces only short factual notes.
- When the combined notes do not fit the final prompt, they are summarized in
  reduction rounds until the complete final prompt fits. Notes are never
  truncated.
- Greedy decoding and fixed seeds make repeated generation deterministic.
- Python normalizes the model response into one fixed structure. The saved
  `generated_summary` contains only the Summary section.


In [ ]:
with open(TEMPLATES_PATH, "r", encoding="utf-8") as f:
    prompt_templates = json.load(f)

single_prompt_template = prompt_templates["single_prompt_template"]
chunk_prompt_template = prompt_templates["chunk_prompt_template"]
notes_reduction_prompt_template = prompt_templates[
    "notes_reduction_prompt_template"
]
final_prompt_template = prompt_templates["final_prompt_template"]

finetuned_model = trainer.model
finetuned_model.eval()
finetuned_model.config.use_cache = True

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
SECTION_PATTERN = re.compile(
    r"(?im)^\s*(Summary|Article Facts|Specialized Vocabulary Explanation|Quiz)\s*:\s*"
)
FACT_LABELS = ("Who", "What", "Where", "When", "Result")


def token_length(text: str, add_special_tokens: bool = True) -> int:
    return len(
        tokenizer(
            str(text),
            add_special_tokens=add_special_tokens,
            truncation=False,
        )["input_ids"]
    )


def split_text_into_chunks(text: str, max_tokens: int):
    units = [
        unit.strip()
        for unit in re.split(r"(?<=[.!?])\s+|\n+", str(text).strip())
        if unit.strip()
    ]

    chunks = []
    current = []

    for unit in units:
        candidate = " ".join(current + [unit]).strip()
        if current and token_length(candidate, False) > max_tokens:
            chunks.append(" ".join(current).strip())
            current = []

        if token_length(unit, False) > max_tokens:
            unit_ids = tokenizer(
                unit,
                add_special_tokens=False,
                truncation=False,
            )["input_ids"]

            if current:
                chunks.append(" ".join(current).strip())
                current = []

            for start in range(0, len(unit_ids), max_tokens):
                piece_ids = unit_ids[start:start + max_tokens]
                piece = tokenizer.decode(
                    piece_ids,
                    skip_special_tokens=True,
                ).strip()
                if piece:
                    chunks.append(piece)
            continue

        current.append(unit)

    if current:
        chunks.append(" ".join(current).strip())

    return chunks if chunks else [str(text).strip()]


def split_article_into_chunks(article: str):
    return split_text_into_chunks(article, CHUNK_ARTICLE_TOKENS)


def generate_response(prompt: str, max_new_tokens: int):
    prompt_tokens = token_length(prompt, add_special_tokens=True)
    if prompt_tokens + max_new_tokens > MAX_CONTEXT:
        raise ValueError(
            f"Generation prompt has {prompt_tokens} tokens and requests "
            f"{max_new_tokens} new tokens, exceeding {MAX_CONTEXT}."
        )

    inputs = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
        return_tensors="pt",
    ).to(DEVICE)
    input_length = inputs["input_ids"].shape[1]

    start_time = time.time()
    with torch.inference_mode():
        output_ids = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start_time

    generated_ids = output_ids[0][input_length:]
    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated_text, elapsed


def clean_factual_notes(text: str) -> str:
    text = re.sub(
        r"(?im)^\s*(?:Factual notes|Shorter factual notes|Summary)\s*:\s*",
        "",
        str(text),
    ).strip()
    return text or "-"


def format_notes(notes):
    return "\n".join(
        f"Part {index}: {note}"
        for index, note in enumerate(notes, start=1)
    )


def build_fitting_final_prompt(notes):
    current_notes = [note for note in notes if str(note).strip()] or ["-"]
    total_time = 0.0
    reduction_rounds = 0

    while True:
        combined = format_notes(current_notes)
        prompt = final_prompt_template.format(chunk_summaries=combined)

        if token_length(prompt, True) <= DIRECT_INPUT_TOKENS:
            return prompt, total_time, reduction_rounds

        note_groups = split_text_into_chunks(
            combined,
            CHUNK_ARTICLE_TOKENS,
        )
        reduced_notes = []

        for group in note_groups:
            reduction_prompt = notes_reduction_prompt_template.format(
                notes=group
            )
            reduced, elapsed = generate_response(
                reduction_prompt,
                max_new_tokens=NOTES_REDUCTION_NEW_TOKENS,
            )
            reduced_notes.append(clean_factual_notes(reduced))
            total_time += elapsed

        current_notes = reduced_notes or ["-"]
        reduction_rounds += 1

        if reduction_rounds > 20:
            raise RuntimeError(
                "The factual notes could not be reduced to a fitting final prompt."
            )


def extract_sections(text: str):
    matches = list(SECTION_PATTERN.finditer(str(text)))
    sections = {}

    for index, match in enumerate(matches):
        name = match.group(1).lower()
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        sections[name] = str(text)[start:end].strip()

    if "summary" not in sections:
        first_header = matches[0].start() if matches else len(str(text))
        sections["summary"] = str(text)[:first_header].strip()

    return sections


def parse_facts(section: str):
    facts = {}
    for label in FACT_LABELS:
        match = re.search(
            rf"(?im)^\s*{label}\s*:\s*(.*)$",
            str(section),
        )
        facts[label] = match.group(1).strip() if match else "-"
        if not facts[label]:
            facts[label] = "-"
    return facts


def parse_quiz(section: str):
    pairs = re.findall(
        r"(?is)(?:^|\n)\s*(?:\d+[.)]\s*)?(?:Question\s*:\s*)?"
        r"(.+?)\s*(?:\n|\s+)\s*Answer\s*:\s*"
        r"(.+?)(?=\n\s*\d+[.)]|\Z)",
        str(section).strip(),
    )

    quiz = []
    for question, answer in pairs[:3]:
        question = re.sub(r"\s+", " ", question).strip(" -")
        answer = re.sub(r"\s+", " ", answer).strip(" -")
        quiz.append((question or "-", answer or "-"))

    while len(quiz) < 3:
        quiz.append(("-", "-"))

    return quiz


def normalize_structured_output(raw_text: str) -> str:
    sections = extract_sections(raw_text)
    summary = re.sub(r"\s+", " ", sections.get("summary", "")).strip() or "-"
    facts = parse_facts(sections.get("article facts", ""))
    vocabulary = sections.get("specialized vocabulary explanation", "").strip() or "-"
    quiz = parse_quiz(sections.get("quiz", ""))

    quiz_text = "\n".join(
        f"{index}. Question: {question}\n   Answer: {answer}"
        for index, (question, answer) in enumerate(quiz, start=1)
    )

    return (
        f"Summary:\n{summary}\n\n"
        "Article Facts:\n"
        f"Who: {facts['Who']}\n"
        f"What: {facts['What']}\n"
        f"Where: {facts['Where']}\n"
        f"When: {facts['When']}\n"
        f"Result: {facts['Result']}\n\n"
        "Specialized Vocabulary Explanation:\n"
        f"{vocabulary}\n\n"
        "Quiz:\n"
        f"{quiz_text}"
    )


def extract_summary_section(generated_text: str) -> str:
    sections = extract_sections(generated_text)
    return re.sub(r"\s+", " ", sections.get("summary", "")).strip()


def generate_article_output(article: str):
    direct_prompt = single_prompt_template.format(article=article)

    if token_length(direct_prompt, True) <= DIRECT_INPUT_TOKENS:
        raw_output, elapsed = generate_response(
            direct_prompt,
            max_new_tokens=MAX_NEW_TOKENS,
        )
        return (
            normalize_structured_output(raw_output),
            raw_output,
            elapsed,
            "direct",
            1,
            0,
        )

    chunks = split_article_into_chunks(article)
    chunk_notes = []
    total_time = 0.0

    for chunk in chunks:
        chunk_prompt = chunk_prompt_template.format(chunk=chunk)
        chunk_output, chunk_time = generate_response(
            chunk_prompt,
            max_new_tokens=CHUNK_NEW_TOKENS,
        )
        chunk_notes.append(clean_factual_notes(chunk_output))
        total_time += chunk_time

    final_prompt, reduction_time, reduction_rounds = build_fitting_final_prompt(
        chunk_notes
    )
    total_time += reduction_time

    raw_output, final_time = generate_response(
        final_prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    total_time += final_time

    return (
        normalize_structured_output(raw_output),
        raw_output,
        total_time,
        "chunked",
        len(chunks),
        reduction_rounds,
    )


In [ ]:
generated_texts = []
raw_generated_texts = []
generation_times = []
generation_methods = []
chunk_counts = []
reduction_round_counts = []

for position, (_, row) in enumerate(test_df.iterrows(), start=1):
    (
        output,
        raw_output,
        elapsed,
        method,
        chunk_count,
        reduction_rounds,
    ) = generate_article_output(row["article"])

    generated_texts.append(output)
    raw_generated_texts.append(raw_output)
    generation_times.append(elapsed)
    generation_methods.append(method)
    chunk_counts.append(chunk_count)
    reduction_round_counts.append(reduction_rounds)

    if position % 10 == 0 or position == len(test_df):
        print(f"Generated {position}/{len(test_df)} responses")


test_df = test_df.copy()
test_df["raw_generated_text"] = raw_generated_texts
test_df["generated_text"] = generated_texts
test_df["generated_summary"] = test_df["generated_text"].apply(
    extract_summary_section
)
test_df["generation_time_sec"] = generation_times
test_df["generation_method"] = generation_methods
test_df["generation_chunk_count"] = chunk_counts
test_df["note_reduction_rounds"] = reduction_round_counts

test_df[
    [
        "generation_method",
        "generation_chunk_count",
        "note_reduction_rounds",
        "summary",
        "generated_summary",
        "generated_text",
    ]
].head()


## 6. Evaluate

ROUGE, readability and compression are calculated from `generated_summary`,
which contains only the Summary section. The complete normalized structure is
checked separately in `generated_text`.


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)


def compute_rouge(reference: str, hypothesis: str):
    scores = scorer.score(reference, hypothesis)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def word_count(text: str) -> int:
    return max(1, len(str(text).split()))


def has_complete_structure(text: str) -> bool:
    required_headers = [
        "Summary:",
        "Article Facts:",
        "Specialized Vocabulary Explanation:",
        "Quiz:",
    ]
    has_headers = all(header in str(text) for header in required_headers)
    has_facts = all(
        re.search(rf"(?m)^{label}:\s*.*$", str(text))
        for label in FACT_LABELS
    )
    has_three_answers = len(
        re.findall(r"(?m)^\s*Answer:\s*.*$", str(text))
    ) == 3
    return bool(has_headers and has_facts and has_three_answers)


rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
flesch_scores = []
compression_ratios = []
structured_complete = []

for _, row in test_df.iterrows():
    rouge = compute_rouge(row["summary"], row["generated_summary"])
    rouge1_scores.append(rouge["rouge1"])
    rouge2_scores.append(rouge["rouge2"])
    rougeL_scores.append(rouge["rougeL"])

    flesch_scores.append(
        textstat.flesch_reading_ease(row["generated_summary"])
        if row["generated_summary"] else 0.0
    )
    compression_ratios.append(
        word_count(row["generated_summary"]) / word_count(row["article"])
    )
    structured_complete.append(has_complete_structure(row["generated_text"]))


test_df["rouge1"] = rouge1_scores
test_df["rouge2"] = rouge2_scores
test_df["rougeL"] = rougeL_scores
test_df["flesch_reading_ease"] = flesch_scores
test_df["compression_ratio"] = compression_ratios
test_df["structured_output_complete"] = structured_complete


## 7. Aggregate and save results

In [ ]:
summary_metrics = pd.DataFrame({
    "metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Flesch Reading Ease",
        "Compression Ratio",
        "Generation Time (sec)",
        "Structured Output Completion Rate",
    ],
    "mean": [
        test_df["rouge1"].mean(),
        test_df["rouge2"].mean(),
        test_df["rougeL"].mean(),
        test_df["flesch_reading_ease"].mean(),
        test_df["compression_ratio"].mean(),
        test_df["generation_time_sec"].mean(),
        test_df["structured_output_complete"].mean(),
    ],
    "std": [
        test_df["rouge1"].std(),
        test_df["rouge2"].std(),
        test_df["rougeL"].std(),
        test_df["flesch_reading_ease"].std(),
        test_df["compression_ratio"].std(),
        test_df["generation_time_sec"].std(),
        test_df["structured_output_complete"].std(),
    ],
})

print(summary_metrics.to_string(index=False))
print("\nGeneration methods:")
print(test_df["generation_method"].value_counts())

per_example_path = os.path.join(RESULTS_DIR, "opt13b_cnn_test_results.csv")
summary_path = os.path.join(RESULTS_DIR, "opt13b_cnn_evaluation_summary.csv")

columns_to_drop = [
    "prompt",
    "training_target",
    "key_points",
    "article_facts",
    "vocabulary",
    "quiz",
]

test_df.drop(
    columns=columns_to_drop,
    errors="ignore",
).to_csv(per_example_path, index=False)
summary_metrics.to_csv(summary_path, index=False)

print(f"\nSaved test results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")
